# Projekt numerika

**Autor:** Marek Slavík 



## Ukol čislo 1 

Naprogramujte v programovacím jazyce Python LUPQ rozklad matice $A ∈ R^{n×n}$ s úplnou

pivotizací, tzn. PAQ = LU, například pomocí následujícího pseudokódu:
```
for k = 1 : n − 1
    Determine µ with k ≤ µ ≤ n and λ with k ≤ λ ≤ n
    so |A(µ, λ)| = max{|A(i, j)| such that i = k : n, j = k : n}
    A(k, 1 : n) ↔ A(µ, 1 : n)
    A(1 : n, k) ↔ A(1 : n, λ)
    p(k) = µ
    g(k) = λ
    if A(k, k) ̸= 0
        rows = (k + 1) : n
        A(rows, k) = A(rows, k)/A(k, k)
        A(rows, rows) = A(rows, rows) − A(rows, k) · A(k, rows)
    end if
end for
```


## MOJE IMPLEMENTACE

In [5]:
import numpy as np
import scipy
from scipy import linalg
import matplotlib.pyplot as plt

In [6]:
def paq(A, n):
    
    # Inicializace polí pro uložení indexů výměn
    p = np.arange(n)
    q = np.arange(n)
    

    for k in range(n - 1):
        # sub_A je matice ve které hledame pivot, začíná na pozici (k, k) a zahrnuje všechny prvky pod a vpravo od této pozice
        sub_A = np.abs(A[k:n, k:n]) 
        
        # np.argmax najde 1D pozici největšího prvku, 
        # np.unravel_index ji převede zpět na 2D souřadnice (řádek, sloupec)
        local_mu, local_lam = np.unravel_index(np.argmax(sub_A), sub_A.shape)
        
        # Přepočet lokálních souřadnic na globální (posuneme je o 'k')
        mu = local_mu + k
        lam = local_lam + k
        
        A[[k, mu], :] = A[[mu, k], :]
        p[[k, mu]] = p[[mu, k]]
        
        # 3. Výměna sloupců (k ↔ λ)
        A[:, [k, lam]] = A[:, [lam, k]]
        q[[k, lam]] = q[[lam, k]]
        
        
        # 5. Gaussova eliminace (pokud pivot není nula)
        if A[k, k] != 0:
            rows = slice(k + 1, n)
            # Výpočet matice L
            A[rows, k] = A[rows, k] / A[k, k]
            # Výpočet matice U
            A[rows, rows] = A[rows, rows] - np.outer(A[rows, k], A[k, rows])
            
    return A, p, q

In [7]:
n = 5 
A = np.random.rand(n, n)

a_vysledna, p, g = paq(A, n)
print("Výsledná matice A (obsahuje matici L pod diagonálou a U na a nad diagonálou):")
print(np.round(a_vysledna, 2))
print("\nŘádkové výměny p:", p)
print("Sloupcové výměny g:", g)

Výsledná matice A (obsahuje matici L pod diagonálou a U na a nad diagonálou):
[[ 0.94  0.64  0.72  0.24  0.09]
 [ 0.11  0.76  0.    0.51  0.48]
 [ 0.17  0.22  0.55  0.14 -0.06]
 [ 0.41  0.22 -0.23  0.51  0.43]
 [ 0.07  0.22  0.01 -0.19  0.32]]

Řádkové výměny p: [4 2 3 1 0]
Sloupcové výměny g: [3 2 1 4 0]


## Ukol čislo 4 

Upravte v programovacím jazyce Python implementaci mocninné metody uvedenou na
přednáškách, resp. na cvičení, tak, aby fungovala pro nalezení nejmenšího vlastního čísla
$λ_1$ SPD matice A. Použijte myšlenku o posunu spektra - tzn. hledejte dominantní vlastní
číslo σn matice B = A − $λ_n I$ , kde $λ_n$ je největší vlastní číslo matice A. Pak $λ_1 = σn + λ_n$
(pozor, zde $σ_n = λ_1 − λ_n < 0$). 

In [89]:


def power_method_dominant(A, x0, max_it=100, tol=1e-10, use_residual_stop=True):
    """
    Mocninná metoda pro nalezení dominantního vlastního čísla (největšího v absolutní hodnotě).
    """
    q = x0 / np.linalg.norm(x0)
    
    for k in range(max_it):
        x = A @ q
        q_new = x / np.linalg.norm(x)
        lambda_k = q_new.T @ A @ q_new

        q = q_new
        
        if use_residual_stop:
            residual = np.linalg.norm(A @ q - lambda_k * q)
            if residual < tol:
                break
    return lambda_k, q

def power_method_smallest_shift(A, x0, max_it=100, tol=1e-10):
    """
    Mocninná metoda s posunem spektra pro nalezení nejmenšího vlastního čísla SPD matice.
    """
    # Najdeme největší vlastní číslo λn
    lambda_n, _ = power_method_dominant(A, x0, max_it, tol)
    
    # Posun spektra: B = A - λn * I
    B = A - lambda_n * np.eye(A.shape[0])
    
    # Najdeme dominantní vlastní číslo σn matice B (σn = λ1 - λn < 0)
    sigma_n, v = power_method_dominant(B, x0, max_it, tol)
    
    # Nejmenší vlastní číslo λ1 = σn + λn
    lambda_1 = sigma_n + lambda_n
    
    return lambda_1, v

In [ ]:
# Testování metody s posunem spektra
n = 5
A_rand = np.random.rand(n, n)
# np .eye(n) přidává n * I, což zajišťuje, že A bude SPD (symetrická a pozitivně definitní)
A = A_rand @ A_rand.T + n * np.eye(n)
x0 = np.random.rand(n)
skutecne_nejmensi = np.min(np.linalg.eigvals(A)).real

lambda_approx_shift, _ = power_method_smallest_shift(A, x0, max_it=100)

print(f"Aproximované nejmenší vl. číslo (posun spektra): {lambda_approx_shift:.6f}")
print(f"Skutečné nejmenší vl. číslo (NumPy):           {skutecne_nejmensi:.6f}")

Aproximované nejmenší vl. číslo (posun spektra): 5.015778
Skutečné nejmenší vl. číslo (NumPy):           5.009637
